# polygon → MOC → shard → coincident waveforms

The waveform half of the minimal read stack. Same two libraries, same polygon,
same public store, zero credentials — but instead of the 3-D scatter, this
notebook does the **cell-level join**: one GEDI o18 footprint against the 2×2
ATL03 o19 cells beneath it, both reconstructed from their stored t-digests as
densities on a shared elevation axis.

It is a separate notebook from `07_minimal` on purpose: that one needs
`%matplotlib widget` for a rotatable 3-D view, this one needs
`%matplotlib inline`, and the two backends collide in a single kernel.

In [ ]:
%pip install -q mortie "moczarr[zagg]>=0.7" matplotlib ipywidgets
%matplotlib inline

import time

import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import Dropdown, FloatText, HBox, IntSlider, VBox, interactive_output

import moczarr as mz
from moczarr.hhdc import rowcol_to_rank
from mortie import generate_morton_children, moc

# The t-digest algebra is IMPORTED from zagg, never vendored (moczarr issue
# #19) -- that is what the `moczarr[zagg]` extra carries, and what
# `mz.read_tensors` has always needed. Two installs, same as 07_minimal.
from zagg.stats.tdigest import cdf_from_tdigest

STORES = {
    "atl03": ("s3://us-west-2.opendata.source.coop/englacial/zagg/demo/atl03_tdigest_o9.zarr", "19/h_tdigest_signal"),
    "gedi": ("s3://us-west-2.opendata.source.coop/englacial/zagg/demo/gedi_flux_o9.zarr", "18/rx_flux"),
}
S3 = {"region": "us-west-2", "anonymous": True}

BLOCK_ORDER = 12
ASIDE, GSIDE = 128, 64  # cells across an o12 block: 2**(19-12) and 2**(18-12)
N_BINS, RES = 256, 1.0  # shared z grid for the paired tensors

## One polygon in, covered shards out

In [ ]:
aoi = {"features": [{"geometry": {"coordinates": [[  # a ~4 km box on the SERC tract
    [-76.56, 38.87], [-76.50, 38.87], [-76.50, 38.91], [-76.56, 38.91], [-76.56, 38.87]
]]}}]}

q = moc(aoi)
shards = None
for name, (root, _field) in STORES.items():
    assert mz.coverage_moc(root, **S3).contains(q), f"{name} does not cover the polygon"
    ids = set(mz.candidate_shards(root, aoi=q, **S3))
    shards = ids if shards is None else shards & ids
shards = sorted(shards)
shards

## Paired tensors — one shard, both sensors

`read_tensors` yields one `(tensor, mask, (offset, gain), block)` per populated
o12 block. Reading both sensors on the same z grid is what makes the two
comparable; the blocks they share are the candidates for a join.

In [ ]:
stores = {name: mz.open_leaf(root, shards[0], **S3) for name, (root, _f) in STORES.items()}

t0 = time.perf_counter()
blocks = {
    name: {b[3]: b for b in mz.read_tensors(
        stores[name], field, n_bins=N_BINS, resolution=RES,
        block_order=BLOCK_ORDER, fit="degrade_resolution")}
    for name, (_root, field) in STORES.items()
}
print(f"paired tensors in {time.perf_counter() - t0:.1f}s — "
      + ", ".join(f"{len(v)} {k} blocks" for k, v in blocks.items()))

# Fold ATL03's o19 footprint down to GEDI's o18 grid, then keep the cells both
# sensors actually populate. SORTED by joint-cell count, densest first, so the
# first pick has real data on both sides.
pairs = []
for w, (gt, _gm, _gz, _) in blocks["gedi"].items():
    if w not in blocks["atl03"]:
        continue
    A2 = blocks["atl03"][w][0].sum(axis=2).reshape(GSIDE, 2, GSIDE, 2).sum(axis=(1, 3))
    G2 = gt.sum(axis=2)
    joint = (A2 > 0) & (G2 > 0)
    if joint.any():
        pairs.append((w, joint, A2, G2))
pairs.sort(key=lambda p: -int(p[1].sum()))

for w, j, A2, G2 in pairs[:8]:
    print(f"  {mz.morton_decimal(w)}  {int(j.sum()):4,} joint o18 cells   "
          f"{int(A2[j].sum()):9,} atl03 photons   {int(G2[j].sum()):9,} gedi pe")

## Coincident waveforms

One GEDI o18 cell against the 2×2 ATL03 o19 cells under it, both read straight
from their stored digests — no tensor in the loop. `cell_index` turns a
chunk-local `(row, col)` into the global cells-axis index `read_cell` wants;
GEDI's chunk grid puts one o12 block in one chunk, while ATL03's chunks sit at
o13, so the ATL03 side resolves which of the block's four o13 children a cell
falls in before addressing it.

The slider ranks joint cells by the **weaker** member — `min(photons, pe)` —
so early picks are genuinely coincident rather than one-sided.

In [ ]:
def _mixture(digest, z, sigma):
    """Digest -> density on `z`: each centroid a Gaussian of width `sigma`."""
    mu, wt = digest[:, 0], digest[:, 1]
    pdf = (wt[None, :] * np.exp(-0.5 * ((z[:, None] - mu[None, :]) / sigma) ** 2)).sum(axis=1)
    return pdf / max(wt.sum(), 1e-9) / (sigma * np.sqrt(2 * np.pi))


def _atl03_digests(w, r, c):
    """The 2x2 o19 digests under one o18 cell `(r, c)` of block `w`."""
    kids = generate_morton_children(int(w), BLOCK_ORDER + 1)  # the block's four o13 chunks
    out = []
    for dr in (0, 1):
        for dc in (0, 1):
            rr, cc = 2 * r + dr, 2 * c + dc          # o19 position inside the block
            chunk = int(kids[rowcol_to_rank(rr // GSIDE, cc // GSIDE, depth=1)])
            try:
                out.append(mz.read_cell(stores["atl03"], STORES["atl03"][1],
                                        mz.cell_index(stores["atl03"], STORES["atl03"][1],
                                                      chunk, rr % GSIDE, cc % GSIDE)))
            except (KeyError, ValueError):
                pass                                  # that quarter holds no photons
    return np.concatenate([k for k in out if len(k)]) if out else np.empty((0, 2))


def paired_waveform(pair=0, nth=0, binw=1.0):
    w, joint, A2, G2 = pairs[pair]
    _, _, (aoff, ag), _ = blocks["atl03"][w]
    _, _, (goff, gg), _ = blocks["gedi"][w]
    rank = np.minimum(A2, G2) * joint            # rank joint cells by the weaker side
    order = np.argsort(rank.ravel())[::-1]
    r, c = np.unravel_index(int(order[min(nth, int(joint.sum()) - 1)]), rank.shape)

    gdigest = mz.read_cell(stores["gedi"], STORES["gedi"][1],
                           mz.cell_index(stores["gedi"], STORES["gedi"][1], int(w), int(r), int(c)))
    adigest = _atl03_digests(w, int(r), int(c))

    lo = min(gdigest[:, 0].min(), adigest[:, 0].min()) - 5
    hi = max(gdigest[:, 0].max(), adigest[:, 0].max()) + 5
    z = np.linspace(lo, hi, 700)
    amu, awt = adigest[:, 0], adigest[:, 1]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.6), sharey=True)
    ax1.plot(_mixture(gdigest, z, gg), z, color="#7b3294", lw=2,
             label=f"GEDI flux ({gdigest[:, 1].sum():.0f} pe)")
    ax1.plot(_mixture(adigest, z, ag), z, color="#008837", lw=2,
             label=f"ATL03 signal ({awt.sum():.0f} ph)")
    ax1.set_xlabel("normalized density")
    ax1.set_ylabel("elevation (m)")

    # Top axis: the RAW ATL03 photons -- binned bars, or one dot per centroid
    # when the width is 0 (cells under the store's delta budget are loss-free,
    # so centroids ~ photons there; a merged centroid's weight sets its size).
    ax1t = ax1.twiny()
    if binw and binw > 0:
        edges = np.arange(lo, hi + binw, binw)
        counts, _ = np.histogram(amu, bins=edges, weights=awt)
        ax1t.barh(edges[:-1] + binw / 2, counts, height=binw * 0.9,
                  color="#008837", alpha=0.25, zorder=0)
        ax1t.set_xlabel(f"ATL03 photons / {binw:g} m bin", fontsize=9)
    else:
        ax1t.scatter(np.zeros(len(amu)), amu, s=np.clip(awt * 8, 8, 40),
                     color="#008837", alpha=0.45, zorder=0)
        ax1t.set_xlim(-0.05, 1.0)
        ax1t.set_xlabel("ATL03 photons (unbinned)", fontsize=9)
    ax1.set_zorder(ax1t.get_zorder() + 1)
    ax1.patch.set_visible(False)
    ax1.set_title(f"cell ({r},{c}) @o18 — GEDI {len(gdigest)} centroids "
                  f"vs ATL03 {len(adigest)} (2×2 @o19)", fontsize=9)
    ax1.legend(fontsize=8)

    # cdf_from_tdigest returns CUMULATIVE WEIGHT (pe for GEDI, photons for
    # ATL03) -- normalize each by its own total so both share the axis honestly.
    ax2.plot(cdf_from_tdigest(gdigest, z) / max(gdigest[:, 1].sum(), 1e-9), z,
             color="#7b3294", lw=2)
    ax2.plot(cdf_from_tdigest(adigest, z) / max(adigest[:, 1].sum(), 1e-9), z,
             color="#008837", lw=2)
    ax2.set_xlim(0, 1)
    ax2.set_xlabel("CDF (probability)")
    ax2.set_title("cumulative", fontsize=10)
    for ax in (ax1, ax2):
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(alpha=0.25, lw=0.5)
    fig.suptitle(f"shard {shards[0]} — block {mz.morton_decimal(w)}", fontsize=11)
    plt.tight_layout()
    plt.show()


_dd = Dropdown(options=[(f"{mz.morton_decimal(w)}  ({int(j.sum()):,} joint cells)", i)
                        for i, (w, j, _, _) in enumerate(pairs)], value=0, description="block")
_nth = IntSlider(min=0, max=40, value=0, description="nth joint")
_binw = FloatText(value=1.0, step=0.5, description="bin (m)")
display(VBox([HBox([_dd, _nth, _binw]),
              interactive_output(paired_waveform, {"pair": _dd, "nth": _nth, "binw": _binw})]))

Two libraries, one polygon — coverage, shards, paired tensors, and the
cell-level waveform join, anonymously against public S3.